In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, avg, stddev, lag, round, current_timestamp
from pyspark.sql.utils import AnalysisException

print("Bắt đầu tính toán chỉ số tài chính (Silver -> Gold)...")

try:
    # 1. Đọc dữ liệu sạch từ lớp Silver
    silver_table = "workspace.stock_db.silver_prices"
    df_silver = spark.read.table(silver_table)
    
    # 2. Định nghĩa các Window (Cửa sổ tính toán)
    window_spec = Window.partitionBy("symbol").orderBy("date")
    window_20 = window_spec.rowsBetween(-19, Window.currentRow)
    window_50 = window_spec.rowsBetween(-49, Window.currentRow)
    
    # 3. Tính toán các chỉ số bằng Window Functions
    df_gold = df_silver \
        .withColumn("prev_close", lag(col("close"), 1).over(window_spec)) \
        .withColumn("daily_return_pct", round(((col("close") - col("prev_close")) / col("prev_close")) * 100, 2)) \
        .withColumn("sma_20", round(avg(col("close")).over(window_20), 2)) \
        .withColumn("sma_50", round(avg(col("close")).over(window_50), 2)) \
        .withColumn("volatility_20d", round(stddev(col("close")).over(window_20), 2)) \
        .withColumn("calculated_at", current_timestamp())
    
    # 4. Lọc bỏ các dòng null 
    df_gold_clean = df_gold.dropna(subset=["daily_return_pct"])
    
    # 5. Chọn các cột cần thiết cho bảng Gold
    final_columns = [
        "symbol", "date", "close", "daily_return_pct", 
        "sma_20", "sma_50", "volatility_20d", "calculated_at"
    ]
    df_final = df_gold_clean.select(*final_columns)
    
    # 6. Ghi dữ liệu vào lớp Gold (Delta Table)
    gold_table = "workspace.stock_db.gold_indicators"
    
    df_final.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .saveAsTable(gold_table)
            
    print(f"--- THÀNH CÔNG: Đã tính toán và lưu chỉ số vào {gold_table} ---")
    
    # 7. Hiển thị kết quả kiểm tra (Xem mã FPT làm ví dụ)
    display(spark.sql(f"SELECT * FROM {gold_table} WHERE symbol = 'FPT.VN' ORDER BY date DESC LIMIT 10"))

except AnalysisException as e:
    print(f"Lỗi: Không tìm thấy bảng Silver. Chi tiết: {e}")
except Exception as e:
    print(f"Đã xảy ra lỗi: {e}")

Databricks visualization. Run in Databricks to view.